# Processing Global-Scale Datasets (CHIRPS, ERA5, TerraClimate)

This notebook demonstrates how to calculate **SPI** and **SPEI** for large global datasets that exceed available RAM.

**Key Features:**
- Memory-efficient chunked processing with spatial tiling
- Automatic memory estimation and chunk size optimization
- Multi-distribution support (Gamma, Pearson III, Log-Logistic)
- Progress tracking for long-running computations
- Streaming I/O to avoid loading entire datasets

**Common Global Datasets:**

| Dataset | Resolution | Grid Size | Approx. Size (monthly, 40+ yr) |
|---------|-----------|-----------|-------------------------------|
| CHIRPS v3 | 0.05 deg | 2400 x 7200 | ~69 GB |
| ERA5 | 0.25 deg | 721 x 1440 | ~4 GB |
| CPC Global | 0.5 deg | 360 x 720 | ~0.7 GB |
| TerraClimate | ~4 km | 4320 x 8640 | ~130 GB |

**Processing Methods:**

| Method | Runs tiles | Saves params | Best For | Section |
|--------|-----------|--------------|----------|---------|
| **Option A** - Simple Global | serial | yes | Most users, quick setup | 4.1 / 5.1 |
| **Option B** - Chunked Processor | serial | yes | Full control, progress tracking | 4.2 / 5.2 |
| **Option C** - Dask Processor | **parallel** | **yes** | Global grids, Zarr or NetCDF output | 4.3 / 5.3 |
| Option D - `compute_index_dask` | parallel | **no** | Lazy graph only, params discarded | - |

> **Use Option C for global TerraClimate.** It is the only route that runs tiles
> concurrently *and* returns the distribution fitting parameters.
> `compute.compute_index_dask()` builds a lazy graph but `map_blocks` discards
> everything except the returned block, so parameters are lost.

---

### Notes from the 2026-09-18 review

Three things were fixed in `src/` after a global SPEI run failed; if you have an
older checkout, pull before running this notebook.

1. **`chunked.py` returned eagerly.** Both chunked functions ended with
   `ds.load()`, which pulls the whole output variable into RAM - 126.8 GiB for
   TerraClimate at 912 x 4320 x 8640 float32 - after the run had already
   finished. Now returns a lazy handle.
2. **Per-chunk writes rewrote the entire file.** The old
   `ds[var].values[...] = tile; ds.to_netcdf(path, mode='a')` materialized the
   full variable on *every* tile and wrote it back to a file that was still
   open. Replaced with a netCDF4 sliced write that touches only the tile.
3. **Pearson III was ~12x slower than it needed to be.** Fitting and transform
   looped cell-by-cell in Python (24.9 million `fit_distribution()` calls per
   1440 x 1440 tile). Method-of-moments Pearson III is just mean/variance/third
   moment, so both are now vectorized. Verified bit-identical against the
   scalar path on 358,520 values including all-NaN, constant, all-zero,
   many-zeros, strictly-negative and near-zero-variance cells.

**Sizing warning:** `n_workers` multiplies peak memory. A 1440 x 1440 SPEI tile
over 912 months peaks near 60 GB, so 40 concurrent workers would need ~2.4 TB.
Use `plan_layout()` in Section 3.4 to pick a tile size that actually fits.

## 1. Setup and Imports

In [1]:
# Add src directory to Python path
import sys
sys.path.insert(0, '../src')

# Core libraries
import numpy as np
import xarray as xr
from pathlib import Path

# Import chunked processing functions (serial tiles)
from chunked import (
    ChunkedProcessor,
    estimate_memory,
    estimate_memory_from_data,
    MemoryEstimate,
    compute_spi_global,
    compute_spei_global
)

# Import Dask processing functions (parallel tiles, keeps fitting params)
from dask_processor import (
    compute_spi_dask,
    compute_spei_dask,
    plan_layout,
    zarr_to_netcdf,
)

# Import standard functions (for comparison / small-data fallback)
from indices import (
    spi,
    spei,
    spi_global,
    spei_global,
    estimate_memory_requirements,
    save_fitting_params,
    load_fitting_params,
    save_index_to_netcdf
)

from compute import compute_index_dask, compute_index_dask_to_zarr
from utils import get_optimal_chunk_size, print_memory_info
from config import Periodicity

print("All imports successful!")

All imports successful!


## 2. Configuration

Set paths and parameters for your global data. Update `BASE_DIR` to point to your data directory.

In [2]:
# ============================================================
# CONFIGURATION
# ============================================================
# Set your base data directory. Works with Windows, WSL, or Linux:
#   Windows:  r"C:\Users\benny\data\global"
#   WSL:      "/mnt/c/Users/benny/data/global"
#   Linux:    "/home/user/data/global"

BASE_DIR = r"C:\Users\benny\OneDrive\Documents\Github\precip-index\global"  # <-- CHANGE THIS
OUTSIDE_ONEDRIVE = r"C:\Users\benny\temp-precip-index"  # <-- CHANGE THIS

# ============================================================
# DERIVED PATHS (no changes needed below)
# ============================================================
base_dir      = Path(BASE_DIR)
outside_onedrive = Path(OUTSIDE_ONEDRIVE)
input_dir     = outside_onedrive / 'input'
output_dir    = outside_onedrive / 'output'
output_netcdf = output_dir / 'netcdf'
output_params = output_dir / 'params'
output_plots  = output_dir / 'plots'

# Create output directories
for d in [output_netcdf, output_params, output_plots]:
    d.mkdir(parents=True, exist_ok=True)

# ============================================================
# INPUT FILES
# ============================================================
# SPI: Precipitation only
# chirps_file = 'wld_cli_chirps3_month1_dekad2_1981_2025.nc'

# SPEI: Precipitation + PET (TerraClimate)
tc_precip_file = 'wld_cli_terraclimate_ppt_1950_2025.nc'
tc_pet_file    = 'wld_cli_terraclimate_pet_1950_2025.nc'

# ============================================================
# CALIBRATION PERIOD (WMO standard: 1991-2020)
# ============================================================
calib_start = 1991
calib_end = 2020

print(f"Base directory:   {base_dir}")
print(f"Input directory:  {input_dir}")
print(f"Output directory: {output_dir}")
print(f"Calibration:      {calib_start}-{calib_end}")
# print(f"\nSPI input:  {chirps_file}")
print(f"SPEI precip: {tc_precip_file}")
print(f"SPEI PET:    {tc_pet_file}")

Base directory:   C:\Users\benny\OneDrive\Documents\Github\precip-index\global
Input directory:  C:\Users\benny\temp-precip-index\input
Output directory: C:\Users\benny\temp-precip-index\output
Calibration:      1991-2020
SPEI precip: wld_cli_terraclimate_ppt_1950_2025.nc
SPEI PET:    wld_cli_terraclimate_pet_1950_2025.nc


## 3. Memory Estimation & Chunk Size

Before processing, always estimate memory requirements to determine if chunking is needed.
The `estimate_memory()` function calculates peak memory usage and recommends chunk sizes.

### 3.1 Estimate Memory Requirements

In [3]:
# Estimate memory for CHIRPS v3 (0.05 deg, Feb 1981 - Dec 2025)
mem_chirps = estimate_memory(
    n_time=540,    # Feb 1981 - Dec 2025 = 539 months
    n_lat=2400,    # CHIRPS v3 global
    n_lon=7200
)
print("CHIRPS v3 (0.05 deg global):")
print(mem_chirps)

print("\n" + "=" * 60)

# Estimate memory for TerraClimate (~4 km, 1950-2025)
mem_tc = estimate_memory(
    n_time=912,    # 68 years * 12 months
    n_lat=4320,    # TerraClimate global
    n_lon=8640
)
print("\nTerraClimate (~4 km global):")
print(mem_tc)

print("\n" + "=" * 60)

# Current system memory
print_memory_info()

2026-09-20 12:28:33 | INFO     | utils | Memory: 33.91 GB used / 382.60 GB total (8.9% used)


CHIRPS v3 (0.05 deg global):
MemoryEstimate(
  Input size: 69.52 GB
  Peak memory needed: 834.27 GB
  Available memory: 348.69 GB
  Status: ✗ Requires chunking
  Recommended chunk size: (2248, 2248) (lat, lon)
  Number of chunks: 8
)


TerraClimate (~4 km global):
MemoryEstimate(
  Input size: 253.62 GB
  Peak memory needed: 3043.43 GB
  Available memory: 348.69 GB
  Status: ✗ Requires chunking
  Recommended chunk size: (1730, 1730) (lat, lon)
  Number of chunks: 15
)



### 3.2 Chunk Size Guidelines

Choose chunk size based on your available RAM:

| Available RAM | Recommended Chunk Size | Notes |
|--------------|------------------------|-------|
| 16 GB | 200 x 200 | Minimum recommended |
| 32 GB | 300 x 300 | Good for workstations |
| 64 GB | 500 x 500 | Standard servers |
| 128 GB | 800 x 800 | High-memory systems |
| 256 GB+ | 1200 x 1200 | Large servers |

**Tips:**
- Larger chunks = faster (fewer I/O operations)
- Smaller chunks = lower memory usage
- Start conservative and increase if stable
- The `get_optimal_chunk_size()` function automates this decision

In [4]:
# Calculate optimal chunk size for different scenarios
print("Optimal chunk sizes for common datasets:")
print("=" * 60)

datasets = [
    ("CHIRPS v3 0.05 deg", 540, 2400, 7200),
    ("ERA5 0.25 deg",      528, 721,  1440),
    ("CPC 0.5 deg",        528, 360,  720),
    ("TerraClimate ~4 km", 912, 4320, 8640),
]

for name, nt, nlat, nlon in datasets:
    chunk_lat, chunk_lon = get_optimal_chunk_size(
        n_time=nt,
        n_lat=nlat,
        n_lon=nlon,
        available_memory_gb=256.0  # Adjust to your system
    )
    print(f"\n  {name}:")
    print(f"    Grid: {nlat} x {nlon}, Time steps: {nt}")
    print(f"    Recommended chunk: ({chunk_lat}, {chunk_lon})")

Optimal chunk sizes for common datasets:

  CHIRPS v3 0.05 deg:
    Grid: 2400 x 7200, Time steps: 540
    Recommended chunk: (1926, 1926)

  ERA5 0.25 deg:
    Grid: 721 x 1440, Time steps: 528
    Recommended chunk: (721, 1440)

  CPC 0.5 deg:
    Grid: 360 x 720, Time steps: 528
    Recommended chunk: (360, 720)

  TerraClimate ~4 km:
    Grid: 4320 x 8640, Time steps: 912
    Recommended chunk: (1482, 1482)


### 3.3 ChunkedProcessor & Progress Callback

The `ChunkedProcessor` is used by Options B throughout this notebook. Configure it once here and reuse for all SPI/SPEI runs.

In [5]:
# Create processor with custom settings
#
# Note: ChunkedProcessor no longer takes n_workers - it walks tiles one at a
# time and the inner computation is vectorized NumPy/SciPy, so there was never
# a worker pool to size. Passing it now raises a DeprecationWarning.
# For concurrent tiles use dask_processor (Sections 4.3 / 5.3).
processor = ChunkedProcessor(
    chunk_lat=1440,   # Process 1440 latitude rows at a time
    chunk_lon=1440,   # Process 1440 longitude columns at a time
    verbose=True      # Print progress information
)

print(f"Processor configured:")
print(f"  Chunk size: {processor.chunk_lat} x {processor.chunk_lon}")
print(f"  Temp dir:   {processor.temp_dir}")

2026-09-20 12:28:35 | INFO     | chunked | ChunkedProcessor initialized: chunk_size=(1440, 1440), tiles processed serially


Processor configured:
  Chunk size: 1440 x 1440
  Temp dir:   C:\TEMP


In [6]:
# Define a progress callback (optional)
def progress_callback(chunk_info, progress_pct):
    """Called after each chunk is processed."""
    if progress_pct % 10 < 2:  # Print every ~10%
        print(f"  [{progress_pct:5.1f}%] {chunk_info}")

### 3.4 Dask Tile Layout Planning

`ChunkedProcessor` runs one tile at a time, so its chunk size only has to fit
once. The Dask path runs `n_workers` tiles **at the same time**, so peak memory
is `n_workers x per_tile_peak`.

`plan_layout()` inverts that: given a worker count and a RAM budget it returns
the largest square tile that still fits. Pass the result as `tile=` and
`n_workers=`, or leave `tile=None` to let the processor plan it for you.

In [7]:
# TerraClimate global SPEI: how tile size has to shrink as workers go up
from config import DASK_TILE_MULTIPLIER_SPEI

print("TerraClimate 912 x 4320 x 8640, SPEI\n" + "=" * 62)
for w in [4, 8, 16, 24, 40]:
    lay = plan_layout(n_time=912, n_lat=4320, n_lon=8640,
                      n_workers=w, index='spei')
    print(f"  {w:3d} workers -> tile {lay.tile:5d}  "
          f"{lay.n_tiles:4d} tiles  "
          f"{lay.per_tile_gb:6.1f} GB/worker  {lay.total_gb:7.1f} GB peak")

print("\nWhat the old notebook asked for (1440 tile, 40 workers):")
bad = plan_layout(912, 4320, 8640, n_workers=40, index='spei')
bad.tile = 1440
bad.per_tile_gb = 1440 * 1440 * 912 * 4 * DASK_TILE_MULTIPLIER_SPEI / 1024**3
bad.total_gb = bad.per_tile_gb * 40
print(bad)

TerraClimate 912 x 4320 x 8640, SPEI
    4 workers -> tile  1223    32 tiles    61.0 GB/worker    243.9 GB peak
    8 workers -> tile   865    50 tiles    30.5 GB/worker    244.0 GB peak
   16 workers -> tile   611   120 tiles    15.2 GB/worker    243.5 GB peak
   24 workers -> tile   499   162 tiles    10.2 GB/worker    243.6 GB peak
   40 workers -> tile   386   276 tiles     6.1 GB/worker    243.0 GB peak

What the old notebook asked for (1440 tile, 40 workers):
DaskLayout(
  Tile size: 1440 x 1440
  Workers: 40   Tiles: 276
  Peak per worker: 84.5 GB
  Peak total: 3381.6 GB / 348.8 GB available [TOO BIG]
)


## 4. SPI Global Processing

SPI requires only **precipitation** data. This section uses CHIRPS v3 global monthly data.

Three processing methods are available --- choose based on your needs:

| Method | Pros | Cons |
|--------|------|------|
| **Option A**: `compute_spi_global()` | Simplest API, minimal code | Less control |
| **Option B**: `ChunkedProcessor` | Full control, progress tracking, param saving | More setup |
| **Option C**: Dask | Lazy evaluation, Zarr output | Requires Dask |

### 4.1 Option A: Simple Global SPI

In [7]:
# ============================================================
# Option A: Simple Global SPI
# ============================================================
# Minimal setup --- just provide file paths and parameters.
# Variable name is auto-detected from the file.
# Uncomment to run:

# result = compute_spi_global(
#     precip_path=str(input_dir / chirps_file),
#     output_path=str(output_netcdf / 'wld_cli_chirps3_d2_spi_gamma_12_month.nc'),
#     scale=12,
#     calibration_start_year=calib_start,
#     calibration_end_year=calib_end,
#     chunk_size=1200,
#     distribution='gamma'
# )

print("Option A: Simple global SPI (uncomment to run)")

Option A: Simple global SPI (uncomment to run)


### 4.2 Option B: Advanced Chunked SPI

In [8]:
# ============================================================
# Option B: Chunked SPI with full control
# ============================================================
# Uses the ChunkedProcessor configured in Section 3.3.
# Uncomment to run:

# result = processor.compute_spi_chunked(
#     precip=str(input_dir / chirps_file),
#     output_path=str(output_netcdf / 'wld_cli_chirps3_d1_spi_gamma_12_month.nc'),
#     scale=12,
#     periodicity='monthly',
#     calibration_start_year=calib_start,
#     calibration_end_year=calib_end,
#     distribution='gamma',         # 'gamma', 'pearson3', or 'log_logistic'
#     save_params=True,             # Save fitting parameters for reuse
#     params_path=str(output_params / 'wld_cli_chirps3_d1_spi_gamma_12_month_params.nc'),
#     compress=True,
#     complevel=4,
#     callback=progress_callback
# )

spi_time_scale = 72
result = processor.compute_spi_chunked(
    precip=str(input_dir / tc_precip_file),
    output_path=str(output_netcdf / f'wld_cli_terraclimate_spi_gamma_{spi_time_scale}_month.nc'),
    scale=spi_time_scale,
    periodicity='monthly',
    calibration_start_year=calib_start,
    calibration_end_year=calib_end,
    distribution='gamma',         # 'gamma', 'pearson3', or 'log_logistic'
    save_params=True,             # Save fitting parameters for reuse
    params_path=str(output_params / f'wld_cli_terraclimate_spi_gamma_{spi_time_scale}_month_params.nc'),
    compress=True,
    complevel=4,
    callback=progress_callback
)

# result = processor.compute_spi_chunked(
#     precip=str(input_dir / chirps_file),
#     output_path=str(output_netcdf / f'wld_cli_chirps3_spi_gamma_{spi_time_scale}_month.nc'),
#     scale=spi_time_scale,
#     periodicity='monthly',
#     calibration_start_year=calib_start,
#     calibration_end_year=calib_end,
#     distribution='gamma',         # 'gamma', 'pearson3', or 'log_logistic'
#     save_params=True,             # Save fitting parameters for reuse
#     params_path=str(output_params / f'wld_cli_chirps3_spi_gamma_{spi_time_scale}_month_params.nc'),
#     compress=True,
#     complevel=4,
#     callback=progress_callback
# )


2026-09-14 16:37:55 | INFO     | chunked | Opening data: C:\Users\benny\temp-precip-index\input\wld_cli_terraclimate_ppt_1950_2025.nc
C:\Users\benny\OneDrive\Documents\Github\precip-index\notebook\../src\chunked.py:342: UserWarning: The specified chunks separate the stored chunks along dimension "lon" starting at index 1440. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_dataset(precip, chunks={'time': -1, 'lat': self.chunk_lat, 'lon': self.chunk_lon})
2026-09-14 16:38:00 | INFO     | chunked | 
MemoryEstimate(
  Input size: 253.62 GB
  Peak memory needed: 3043.43 GB
  Available memory: 348.24 GB
  Status: ✗ Requires chunking
  Recommended chunk size: (1729, 1729) (lat, lon)
  Number of chunks: 15
)
2026-09-14 16:38:00 | INFO     | chunked | Creating output file: C:\Users\benny\temp-precip-index\output\netcdf\wld_cli_terraclimate_spi_gamma_72_month.nc
2026-09-14 16:56:23 | INFO     | chunked | Processing 18 chunks with size (1440, 1440)
2026-

  [ 11.1%] Chunk 2/18 [lat 0:1440, lon 1440:2880]


2026-09-14 18:26:35 | INFO     | compute | Computing index: shape=(912, 1440, 1440), scale=72, distribution=gamma, grid_cells=2,073,600
2026-09-14 18:26:39 | INFO     | compute | Step 1/3: Applying temporal scaling...
2026-09-14 18:28:38 | INFO     | compute | Step 2/3: Computing gamma parameters...
2026-09-14 18:28:58 | INFO     | compute | Step 3/3: Transforming to standard normal...
2026-09-14 18:33:08 | INFO     | compute | Index computation complete
2026-09-14 19:05:34 | INFO     | chunked | Processing Chunk 4/18 [lat 0:1440, lon 4320:5760] (22.2%)
2026-09-14 19:09:38 | INFO     | compute | Computing index: shape=(912, 1440, 1440), scale=72, distribution=gamma, grid_cells=2,073,600
2026-09-14 19:09:42 | INFO     | compute | Step 1/3: Applying temporal scaling...
2026-09-14 19:11:45 | INFO     | compute | Step 2/3: Computing gamma parameters...
2026-09-14 19:12:05 | INFO     | compute | Step 3/3: Transforming to standard normal...
2026-09-14 19:18:10 | INFO     | compute | Index co

  [ 50.0%] Chunk 9/18 [lat 1440:2880, lon 2880:4320]


2026-09-14 23:59:09 | INFO     | compute | Computing index: shape=(912, 1440, 1440), scale=72, distribution=gamma, grid_cells=2,073,600
2026-09-14 23:59:13 | INFO     | compute | Step 1/3: Applying temporal scaling...
2026-09-15 00:01:10 | INFO     | compute | Step 2/3: Computing gamma parameters...
2026-09-15 00:01:29 | INFO     | compute | Step 3/3: Transforming to standard normal...
2026-09-15 00:08:46 | INFO     | compute | Index computation complete
2026-09-15 00:48:15 | INFO     | chunked | Processing Chunk 11/18 [lat 1440:2880, lon 5760:7200] (61.1%)
2026-09-15 00:51:47 | INFO     | compute | Computing index: shape=(912, 1440, 1440), scale=72, distribution=gamma, grid_cells=2,073,600
2026-09-15 00:51:51 | INFO     | compute | Step 1/3: Applying temporal scaling...
2026-09-15 00:53:49 | INFO     | compute | Step 2/3: Computing gamma parameters...
2026-09-15 00:54:08 | INFO     | compute | Step 3/3: Transforming to standard normal...
2026-09-15 00:58:34 | INFO     | compute | Inde

  [ 61.1%] Chunk 11/18 [lat 1440:2880, lon 5760:7200]


2026-09-15 01:43:10 | INFO     | compute | Computing index: shape=(912, 1440, 1440), scale=72, distribution=gamma, grid_cells=2,073,600
2026-09-15 01:43:14 | INFO     | compute | Step 1/3: Applying temporal scaling...
2026-09-15 01:45:12 | INFO     | compute | Step 2/3: Computing gamma parameters...
2026-09-15 01:45:31 | INFO     | compute | Step 3/3: Transforming to standard normal...
2026-09-15 01:49:25 | INFO     | compute | Index computation complete
2026-09-15 02:30:35 | INFO     | chunked | Processing Chunk 13/18 [lat 2880:4320, lon 0:1440] (72.2%)
2026-09-15 02:33:50 | INFO     | compute | Computing index: shape=(912, 1440, 1440), scale=72, distribution=gamma, grid_cells=2,073,600
2026-09-15 02:33:53 | INFO     | compute | Step 1/3: Applying temporal scaling...
2026-09-15 02:35:52 | INFO     | compute | Step 2/3: Computing gamma parameters...
2026-09-15 02:36:11 | INFO     | compute | Step 3/3: Transforming to standard normal...
2026-09-15 02:40:08 | INFO     | compute | Index c

  [100.0%] Chunk 18/18 [lat 2880:4320, lon 7200:8640]


2026-09-15 07:59:01 | INFO     | indices | Fitting parameters saved: C:\Users\benny\temp-precip-index\output\params\wld_cli_terraclimate_spi_gamma_72_month_params.nc
2026-09-15 07:59:01 | INFO     | chunked | Chunked SPI computation complete: C:\Users\benny\temp-precip-index\output\netcdf\wld_cli_terraclimate_spi_gamma_72_month.nc


MemoryError: Unable to allocate 127. GiB for an array with shape (912, 4320, 8640) and data type float32

### 4.3 Option C: Dask SPI (parallel tiles, saves parameters)

`compute_spi_dask()` submits each spatial tile to a Dask worker. Every worker
opens the input file itself, reads only its own tile, computes SPI, writes its
own slice of the output, and returns just the fitting parameters to the client.
The full grid is never held in memory anywhere.

| `output_format` | Writes | Parallel writes | Resumable | Needs |
|---|---|---|---|---|
| `'zarr'` | disjoint regions of a Zarr store | yes, lock-free | yes - finished tiles survive a crash | `zarr` |
| `'netcdf'` | slices of a pre-allocated `.nc` | serialized by a lock | no | `netCDF4` |

Use `'zarr'` for a first global run, then `zarr_to_netcdf()` to stream out a
`.nc` if you need one.

In [ ]:
# ============================================================
# Option C: Dask SPI - parallel tiles, fitting params saved
# ============================================================
# Uncomment to run.

# spi_scale = 12
#
# out_zarr = compute_spi_dask(
#     precip_path=str(input_dir / tc_precip_file),
#     output_path=str(output_netcdf / f'wld_cli_terraclimate_spi_gamma_{spi_scale}_month.zarr'),
#     scale=spi_scale,
#     periodicity='monthly',
#     calibration_start_year=calib_start,
#     calibration_end_year=calib_end,
#     distribution='gamma',
#     output_format='zarr',       # or 'netcdf'
#     tile=None,                  # None -> planned from n_workers and free RAM
#     n_workers=8,
#     memory_limit='40GB',        # per worker
#     save_params=True,
#     params_path=str(output_params / f'wld_cli_terraclimate_spi_gamma_{spi_scale}_month_params.nc'),
# )
#
# # Optional: stream the Zarr store out to NetCDF
# zarr_to_netcdf(
#     out_zarr,
#     str(output_netcdf / f'wld_cli_terraclimate_spi_gamma_{spi_scale}_month.nc'),
# )

print("Option C: Dask SPI (uncomment to run)")

### 4.4 Reusing SPI Parameters

For repeated calculations with the same calibration period (e.g., updating with new months of data),
load previously saved parameters to skip the fitting step.

In [ ]:
# Step 1: First run --- save parameters using save_params=True (any method above)
# The chunked processor automatically saves params when save_params=True.

# Step 2: Load previously saved parameters
# params = load_fitting_params(
#     str(output_params / 'wld_cli_chirps3_d2_spi_gamma_12_month_params.nc'),
#     scale=12,
#     periodicity='monthly'
# )
# print(f"Loaded parameters: {list(params.keys())}")
# for k, v in params.items():
#     if hasattr(v, 'shape'):
#         print(f"  {k}: shape={v.shape}")

# Step 3: Re-run with pre-computed parameters (much faster)
# result_fast = processor.compute_spi_chunked(
#     precip=str(input_dir / 'updated_precip_data.nc'),
#     output_path=str(output_netcdf / 'spi_12_updated.nc'),
#     scale=12,
#     calibration_start_year=calib_start,
#     calibration_end_year=calib_end,
#     distribution='gamma',
#     save_params=False  # Already have params
# )

print("Parameter reuse workflow (uncomment to run)")

## 5. SPEI Global Processing

SPEI requires both **precipitation** and **PET** (Potential Evapotranspiration) data.
This section uses TerraClimate global monthly data (1958--2024).

**Input files** (in the same `input` directory):

| File | Variable | Period |
|:-----|:---------|:-------|
| `wld_cli_terraclimate_ppt_1950_2025.nc` | `ppt` | 1958--2024 |
| `wld_cli_terraclimate_pet_1950_2025.nc` | `pet` | 1958--2024 |

Variable names (`ppt`, `pet`) are auto-detected by the processor.

### 5.1 Option A: Simple Global SPEI

In [ ]:
# ============================================================
# Option A: Simple Global SPEI
# ============================================================
# Minimal setup --- just provide precip and PET file paths.
# Uncomment to run:

# --- SPEI-12 with Gamma distribution ---
# result_spei = compute_spei_global(
#     precip_path=str(input_dir / tc_precip_file),
#     pet_path=str(input_dir / tc_pet_file),
#     output_path=str(output_netcdf / 'wld_cli_terraclimate_spei_gamma_12_month.nc'),
#     scale=12,
#     calibration_start_year=calib_start,
#     calibration_end_year=calib_end,
#     chunk_size=1200,
#     distribution='gamma'
# )

# --- SPEI-12 with Pearson III (recommended for SPEI) ---
# result_spei_p3 = compute_spei_global(
#     precip_path=str(input_dir / tc_precip_file),
#     pet_path=str(input_dir / tc_pet_file),
#     output_path=str(output_netcdf / 'wld_cli_terraclimate_spei_pearson3_12_month.nc'),
#     scale=12,
#     calibration_start_year=calib_start,
#     calibration_end_year=calib_end,
#     chunk_size=1200,
#     distribution='pearson3'
# )

print("Option A: Simple global SPEI (uncomment to run)")

### 5.2 Option B: Advanced Chunked SPEI

Uses the same `ChunkedProcessor` configured in Section 3.3.

In [ ]:
# ============================================================
# Option B: Chunked SPEI (serial tiles)
# ============================================================
# Works again after the chunked.py fixes, but processes one tile at a time.
# For global TerraClimate prefer Option C (Section 5.3) - same output, tiles
# computed concurrently.
#
# If you do run this, note that `processor` in Section 3.3 is configured with
# chunk 1440 x 1440. That is fine here (serial, one tile live at a time) but
# is NOT a safe Dask tile size at 40 workers.
#
# Uncomment to run:

spei_time_scale = 1

result_spei_p3 = processor.compute_spei_chunked(
    precip=str(input_dir / tc_precip_file),
    pet=str(input_dir / tc_pet_file),
    output_path=str(output_netcdf / f'wld_cli_terraclimate_spei_pearson3_{spei_time_scale}_month.nc'),
    scale=spei_time_scale,
    periodicity='monthly',
    calibration_start_year=calib_start,
    calibration_end_year=calib_end,
    distribution='pearson3',
    save_params=True,
    params_path=str(output_params / f'wld_cli_terraclimate_spei_pearson3_{spei_time_scale}_month_params.nc'),
    compress=True,
    complevel=4,
    callback=progress_callback
)

# The return is a LAZY handle - do not call .load() on a global grid
print(result_spei_p3)

# print("Option B: Chunked SPEI (uncomment to run; prefer Option C for global)")

2026-09-19 22:07:50 | INFO     | chunked | Opening precipitation data
C:\Users\benny\OneDrive\Documents\Github\precip-index\notebook\../src\chunked.py:698: UserWarning: The specified chunks separate the stored chunks along dimension "lon" starting at index 1440. This could degrade performance. Instead, consider rechunking after loading.
  precip_ds = xr.open_dataset(precip, chunks={'time': -1, 'lat': self.chunk_lat, 'lon': self.chunk_lon})
2026-09-19 22:07:50 | INFO     | chunked | Opening PET data
C:\Users\benny\OneDrive\Documents\Github\precip-index\notebook\../src\chunked.py:714: UserWarning: The specified chunks separate the stored chunks along dimension "lon" starting at index 1440. This could degrade performance. Instead, consider rechunking after loading.
  pet_ds = xr.open_dataset(pet, chunks={'time': -1, 'lat': self.chunk_lat, 'lon': self.chunk_lon})
2026-09-19 22:07:50 | INFO     | chunked | 
MemoryEstimate(
  Input size: 253.62 GB
  Peak memory needed: 4565.15 GB
  Available

### 5.3 Option C: Dask SPEI (parallel tiles, saves parameters)

**This is the recommended route for global TerraClimate SPEI.**

Each worker reads its own precipitation and PET tile, forms the water balance
`(P - PET) + offset` locally, fits Pearson III, writes its slice, and returns
only the parameter tile. Nothing ever holds the 126.8 GiB output array.

#### Resuming

Every completed tile is recorded in `<output>.manifest.json`, and its fitting
parameters land in `<output>_tileparams/` as the tile finishes. So:

- **If the run dies, re-run the same cell.** It reuses the existing store and
  computes only the missing tiles.
- A tile that raises no longer aborts the other 49 - the failure is logged,
  the remaining tiles finish, and the run ends by telling you which tiles to retry.
- The manifest is only honoured when scale, distribution, calibration, grid
  shape and tile size all match. Change any of them and it recomputes cleanly.
- `resume=False` starts over and **erases** the existing store.

#### Sizing

Run Section 3.4 first. Remember peak memory is `n_workers x per_tile_peak`, so
more workers needs *smaller* tiles, not bigger ones:

| Workers | Tile | Tiles | Per worker | Total |
|--------:|-----:|------:|-----------:|------:|
| 6 | 640 | 98 | ~16 GB | ~98 GB |
| 8 | 912 | 50 | ~31 GB | ~249 GB |
| 16 | 645 | 98 | ~16 GB | ~249 GB |

The per-tile multiplier is an estimate, not a measurement, so start conservative
and raise it once a full run has succeeded.

In [ ]:
# ============================================================
# Option C: Dask SPEI - parallel tiles, fitting params saved
# ============================================================
# RESUMABLE. Each finished tile is recorded in <output>.manifest.json and its
# fitting parameters are written to <output>_tileparams/. If this dies, just
# re-run this same cell: it skips completed tiles and continues.
# resume=False starts over and ERASES the existing store.
spei_scale = 6

out_store = compute_spei_dask(
    precip_path=str(input_dir / tc_precip_file),
    pet_path=str(input_dir / tc_pet_file),
    output_path=str(output_netcdf / f'wld_cli_terraclimate_spei_pearson3_{spei_scale}_month.zarr'),
    scale=spei_scale,
    periodicity='monthly',
    calibration_start_year=calib_start,
    calibration_end_year=calib_end,
    distribution='pearson3',
    output_format='zarr',

    # Conservative sizing. The 8-worker / 911-tile attempt died at tile 12 of
    # 50 for a reason the cluster-shutdown error hid. 640 x 640 across 6
    # workers is ~16 GB per worker and ~98 GB total instead of ~249 GB, and
    # 98 smaller tiles means a failure costs less and resumes more finely.
    # Raise these once a full run has succeeded.
    tile=640,
    n_workers=6,
    memory_limit='40GB',        # per worker

    save_params=True,
    params_path=str(output_params / f'wld_cli_terraclimate_spei_pearson3_{spei_scale}_month_params.nc'),
    resume=True,
    dashboard=True,             # link is printed in the log
)

print(f"\nWrote: {out_store}")

2026-09-20 12:28:48 | INFO     | dask_processor | 
DaskLayout(
  Tile size: 640 x 640
  Workers: 6   Tiles: 98
  Peak per worker: 16.7 GB
  Peak total: 100.2 GB / 348.7 GB available [ok]
)
2026-09-20 12:28:48 | INFO     | dask_processor | Creating zarr template: C:\Users\benny\temp-precip-index\output\netcdf\wld_cli_terraclimate_spei_pearson3_6_month.zarr
C:\Users\benny\miniforge3\envs\climate\Lib\site-packages\zarr\api\asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(
2026-09-20 12:28:49 | INFO     | dask_processor | Starting Dask cluster: 6 workers, 98 tiles
2026-09-20 12:28:50 | INFO     | distributed.http.proxy | To route to workers diagnostics web server please install jupyter-server-proxy: python -m pip install jupyter-server-proxy
2026-09-20 12:28:50 | INFO     | distributed.scheduler | State start
2026-09-20 12:

In [ ]:
# Stream the Zarr store out to a single NetCDF file (chunk-by-chunk, no full load)
nc_path = zarr_to_netcdf(
    out_store,
    str(output_netcdf / f'wld_cli_terraclimate_spei_pearson3_{spei_scale}_month.nc'),
    compress=True,
    complevel=4,
)

# Check the result without pulling the whole grid into memory
ds = xr.open_dataset(nc_path)
var = list(ds.data_vars)[0]
print(ds[var])
sample = ds[var].isel(time=-1, lat=slice(0, None, 20), lon=slice(0, None, 20)).values
print(f"\nlast month, every 20th cell: finite={np.isfinite(sample).sum():,} "
      f"mean={np.nanmean(sample):.3f} std={np.nanstd(sample):.3f}")
ds.close()

### 5.4 Reusing SPEI Parameters

Same workflow as SPI --- save parameters once, reload for subsequent runs.

In [ ]:
# Load previously saved SPEI parameters (works the same whether they were
# produced by ChunkedProcessor or by compute_spei_dask)
spei_scale = 12
params_file = output_params / f'wld_cli_terraclimate_spei_pearson3_{spei_scale}_month_params.nc'

if params_file.exists():
    params_spei = load_fitting_params(
        str(params_file), scale=spei_scale, periodicity='monthly'
    )
    print(f"distribution: {params_spei.get('distribution')}")
    for k, v in params_spei.items():
        if hasattr(v, 'shape'):
            print(f"  {k:10s} shape={v.shape}  finite={np.isfinite(v).sum():,}")
else:
    print(f"Not found yet: {params_file.name}")
    print("Run Section 5.3 with save_params=True first.")

# Pearson III stores skew / loc / scale / prob_zero.
# Note: prob_zero is recorded but is NOT applied in the Pearson III transform
# (the Gamma path does apply it). Kept as-is so results stay reproducible.

## 6. Batch Processing: Multi-Scale & Multi-Distribution

For comprehensive analysis, you may want to compute indices at multiple scales
(e.g., 1, 3, 6, 12 months) and/or with multiple distributions (Gamma, Pearson III,
Log-Logistic). This section shows how to automate these combinations.

Each combination produces a separate output file. Processing time scales linearly
with the number of combinations.

### 6.1 Multi-Scale, Single Distribution

In [ ]:
# ============================================================
# Compute SPI at multiple scales with Gamma distribution
# ============================================================
# Uncomment to run:

# scales = [1, 3, 6, 12]
# distribution = 'gamma'
#
# for scale in scales:
#     print(f"\n{'='*60}")
#     print(f"Processing SPI-{scale} ({distribution})")
#     print(f"{'='*60}")
#
#     processor.compute_spi_chunked(
#         precip=str(input_dir / chirps_file),
#         output_path=str(output_netcdf / f'wld_cli_chirps3_d2_spi_{distribution}_{scale}_month.nc'),
#         scale=scale,
#         periodicity='monthly',
#         calibration_start_year=calib_start,
#         calibration_end_year=calib_end,
#         distribution=distribution,
#         save_params=True,
#         params_path=str(output_params / f'wld_cli_chirps3_d2_spi_{distribution}_{scale}_month_params.nc'),
#         compress=True,
#         complevel=4,
#         callback=progress_callback
#     )
#
# print(f"\nCompleted SPI at scales: {scales}")

print("Multi-scale SPI batch processing (uncomment to run)")

### 6.2 Single Scale, Multiple Distributions

In [ ]:
# ============================================================
# Compute SPI-12 with all supported distributions
# ============================================================
# Uncomment to run:

# scale = 12
# distributions = ['gamma', 'pearson3', 'log_logistic']
#
# for dist in distributions:
#     print(f"\n{'='*60}")
#     print(f"Processing SPI-{scale} ({dist})")
#     print(f"{'='*60}")
#
#     processor.compute_spi_chunked(
#         precip=str(input_dir / chirps_file),
#         output_path=str(output_netcdf / f'wld_cli_chirps3_d2_spi_{dist}_{scale}_month.nc'),
#         scale=scale,
#         periodicity='monthly',
#         calibration_start_year=calib_start,
#         calibration_end_year=calib_end,
#         distribution=dist,
#         save_params=True,
#         params_path=str(output_params / f'wld_cli_chirps3_d2_spi_{dist}_{scale}_month_params.nc'),
#         compress=True,
#         complevel=4,
#         callback=progress_callback
#     )
#
# print(f"\nCompleted SPI-{scale} with distributions: {distributions}")

print("Multi-distribution SPI batch processing (uncomment to run)")

### 6.3 Multi-Scale, Multiple Distributions

In [ ]:
# ============================================================
# Full combination: multiple scales x multiple distributions
# ============================================================
# WARNING: This produces many output files and can take a long time.
# For example, 4 scales x 3 distributions = 12 separate runs.
# Uncomment to run:

# scales = [3, 6, 12]
# distributions = ['gamma', 'pearson3', 'log_logistic']
#
# total = len(scales) * len(distributions)
# run = 0
#
# for scale in scales:
#     for dist in distributions:
#         run += 1
#         print(f"\n{'='*60}")
#         print(f"[{run}/{total}] SPI-{scale} ({dist})")
#         print(f"{'='*60}")
#
#         processor.compute_spi_chunked(
#             precip=str(input_dir / chirps_file),
#             output_path=str(output_netcdf / f'wld_cli_chirps3_d2_spi_{dist}_{scale}_month.nc'),
#             scale=scale,
#             periodicity='monthly',
#             calibration_start_year=calib_start,
#             calibration_end_year=calib_end,
#             distribution=dist,
#             save_params=True,
#             params_path=str(output_params / f'wld_cli_chirps3_d2_spi_{dist}_{scale}_month_params.nc'),
#             compress=True,
#             complevel=4,
#             callback=progress_callback
#         )
#
# print(f"\nCompleted all {total} combinations")

print("Full batch processing (uncomment to run)")

### 6.4 Batch SPEI (Multi-Scale / Multi-Distribution)

The same loop patterns work for SPEI --- just replace `compute_spi_chunked` with
`compute_spei_chunked` and provide both precipitation and PET files.

In [ ]:
# ============================================================
# Batch SPEI: multiple scales with Pearson III
# ============================================================
# Uncomment to run:

# scales = [3, 6, 12]
# distribution = 'pearson3'
#
# for scale in scales:
#     print(f"\n{'='*60}")
#     print(f"Processing SPEI-{scale} ({distribution})")
#     print(f"{'='*60}")
#
#     processor.compute_spei_chunked(
#         precip=str(input_dir / tc_precip_file),
#         pet=str(input_dir / tc_pet_file),
#         output_path=str(output_netcdf / f'wld_cli_terraclimate_spei_{distribution}_{scale}_month.nc'),
#         scale=scale,
#         periodicity='monthly',
#         calibration_start_year=calib_start,
#         calibration_end_year=calib_end,
#         distribution=distribution,
#         save_params=True,
#         params_path=str(output_params / f'wld_cli_terraclimate_spei_{distribution}_{scale}_month_params.nc'),
#         compress=True,
#         complevel=4,
#         callback=progress_callback
#     )
#
# print(f"\nCompleted SPEI at scales: {scales}")

print("Batch SPEI processing (uncomment to run)")

## 7. Performance Tips

1. **Use SSD storage** --- Significantly faster I/O for chunked processing
2. **Pre-process data** --- Ensure data is already in (time, lat, lon) order
3. **Compress output** --- Use `compress=True` to reduce file sizes by 50--70%
4. **Monitor memory** --- Watch for memory leaks during long runs
5. **Use Zarr for large outputs** --- Better for parallel writes than NetCDF
6. **Choose the right method:**
   - **Option A** (Simple) --- Best for most users, minimal code
   - **Option B** (Chunked) --- When you need progress tracking or custom chunks
   - **Option C** (Dask) --- When data doesn't fit even with chunking
7. **Reuse parameters** --- After first run with `save_params=True`, subsequent runs skip the expensive fitting step
8. **Distribution defaults** --- Gamma uses Method of Moments; Pearson III uses Method of Moments; Log-Logistic uses MLE

In [ ]:
# Monitor memory usage during processing
print_memory_info()

## 8. Summary

### Decision Flowchart

```
Does your data fit in RAM?
├── YES → spi() / spei() from indices   (notebooks 01-02)
│
└── NO → Do you need the fitting parameters back?
         ├── YES, and you want tiles run in parallel
         │        → compute_spi_dask / compute_spei_dask   (4.3 / 5.3)
         │          output_format='zarr'   parallel, resumable
         │          output_format='netcdf' locked writes, single .nc
         │
         ├── YES, serial is fine
         │        → ChunkedProcessor                        (4.2 / 5.2)
         │          or compute_spi_global / compute_spei_global (4.1 / 5.1)
         │
         └── NO, index values only
                  → compute_index_dask / compute_index_dask_to_zarr
                    (params are discarded by map_blocks)
```

### Available Distributions

| Distribution | Default Method | Vectorized | Best For |
|:------------|:--------------|:-----------|:---------|
| Gamma | Method of Moments | yes | SPI (standard choice) |
| Pearson III | Method of Moments | yes (since 2026-09-18) | SPEI, skewed data |
| Log-Logistic | MLE | no - per-cell loop | Better tail behavior |
| GEV, Gen. Logistic | L-moments | no - per-cell loop | Extremes |

Gamma and Pearson III both run vectorized. The remaining distributions still
use the per-cell path, so expect them to be roughly an order of magnitude
slower on a global grid.

### Requirements

`dask`, `distributed` and `zarr` are needed for the Dask route (and `dask` is
needed by `ChunkedProcessor` too, since it opens inputs with `chunks=`).
`psutil` is what lets the memory planners see real free RAM; without it they
assume 16 GB and will under-size every tile.

### Next Steps

- **Notebook 01**: Calculate SPI with multiple distributions
- **Notebook 02**: Calculate SPEI with PET options
- **Notebook 03**: Analyze climate extreme events using run theory
- **Notebook 04**: Explore the visualization gallery